In [ ]:
# ================================================================
# Import required libraries
# ================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pathlib import Path
from skimage import io
from skimage.io import imread_collection
import datetime
import re
import builtins
import os
import imageio.v2 as imageio
from IPython.display import Image, display
from collections import defaultdict

In [ ]:
# ================================================================
# Configuration
# ================================================================

# INPUT experiment name
experiment_name = ""


# Path specification

base_path = Path("/mnt/large_storage/shared_storage/data/Vicen")
data_path = base_path / "data" / experiment_name 

output_path = base_path / "extracted_pngs" / experiment_name
output_path.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    raise FileNotFoundError(f"data_path does not exist: {data_path}")

print(f"  data_path        = {data_path}")
print(f"  experiment_name  = {experiment_name}")
print(f"  output_path      = {output_path}")

In [ ]:

# File discovery

image_patterns = ["*.tif", "*.tiff", "*.ome.tif", "*.ome.tiff"]
filenames = []
for pattern in image_patterns:
    filenames.extend(data_path.rglob(pattern)) # recursively search for image files

filenames = sorted({str(f) for f in filenames})

print(f"Loaded {len(filenames)} images \n")

# Diagnostic: check file counts and timestamp uniqueness per LED/FOV

files_by_led = defaultdict(list)
files_by_led_by_fov = defaultdict(lambda: defaultdict(list))


for f in filenames:
    led = get_led(f)
    fov = get_fov(f)
    files_by_led[led].append(f)
    files_by_led_by_fov[led][fov].append(f)

print(f"Files per LED:")

for led in sorted(files_by_led):
    print(f"{led}: {len(files_by_led[led])} files")
    # for fov in sorted(files_by_led_by_fov[led]):
    #     print(f"  FOV {fov}: {len(files_by_led_by_fov[led][fov])} files")

In [ ]:
# ================================================================
# Assign folder names
# ================================================================

# images = [io.imread(p) for p in filenames]
# image_names = [Path(p).parent.name for p in filenames]
# 
# print(f"Assigned name to {len(images)} images")
# for idx, (name, path) in enumerate(zip(image_names, filenames)):
#     print(f"{idx}: {name}")

In [ ]:
# INPUT LED selection (multiple may be selected, e.g., ["LED515NM", "LEDOVERHEADTIGER"])

#sel_led = ["LED515NM", "LEDOVERHEADTIGER"]
sel_led = ["LED565NM"]

# Select LED
for led in sel_led:
    if not (led.startswith('LED') and (led.endswith('NM') or led.endswith('OVERHEAD') or led.endswith('OVERHEADTIGER'))):
        raise ValueError(f"Invalid LED format: {led}. Expected format: LED<wavelength>NM or LEDOVERHEAD")

print(f"Selected LEDs = {sel_led}")

# Group filenames by position for the selected LEDs
fovs_by_led = {}
filenames_by_fov_by_led = {}

for led in sel_led:
    fovs_by_led[led] = np.unique([get_fov(f) for f in filenames if get_led(f) == led])
    filenames_by_fov_by_led[led] = {
        fov: sort_by_time([f for f in filenames if get_led(f) == led and get_fov(f) == fov])
        for fov in fovs_by_led[led]
    }

# Load images per position per LED
imgs_by_led = {}

for led in sel_led:
    imgs_by_led[led] = []
    for fov in fovs_by_led[led]:
        if filenames_by_fov_by_led[led][fov]:
            imgs_by_led[led].append(imread_collection(filenames_by_fov_by_led[led][fov]))


# Diagnostic: check file counts and timestamp uniqueness per LED/FOV

for led in sel_led:
    print(f"\n{led} - {len(fovs_by_led[led])} total files found accross {len(fovs_by_led[led])} FOVs")
    for fov in fovs_by_led[led]:
        files = filenames_by_fov_by_led[led][fov]
        if len(files) <= len(fovs_by_led[led]):
             raise ValueError(
                f"User selected {led}, but this LED has only one file per FOV. "
                f"The LED was likely only used for initialisation and did not image over time")
        print(f"  FOV {fov}: files={len(files)}")

In [ ]:
# INPUT image adjustment parameters

fov_to_display = 2
v_min_perc = [0]
v_max_perc = [100]

#y_min, y_max, x_min, x_max = 0, 30000, 0, 3000
y_min, y_max, x_min, x_max = 0, 3200, 1100, 1700

if len(v_min_perc) != len(sel_led) or len(v_max_perc) != len(sel_led):
    raise ValueError(f"Expected {len(sel_led)} values in v_min_perc and v_max_perc - one per selected LED")

print(f"Plotting FOV no {fov_to_display}")

# Display cropped images with raw and adjusted contrast per LED

for i, led in enumerate(sel_led):
    if fov_to_display not in filenames_by_fov_by_led[led]:
        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].set_axis_off()
        axes[1].set_axis_off()
        fig.suptitle(led)
        axes[0].set_title("FOV not found")
        plt.tight_layout()
        plt.show()
        continue

    first_file = filenames_by_fov_by_led[led][fov_to_display][0]
    img = io.imread(first_file)[y_min:y_max, x_min:x_max]
    v_min = np.percentile(img, v_min_perc[i])
    v_max = np.percentile(img, v_max_perc[i])

    fig, axes = plt.subplots(1, 2, figsize=(4, 5))
    fig.suptitle(led)
    axes[0].imshow(img, cmap="viridis")
    axes[0].set_title("raw")

    axes[1].imshow(img, cmap="viridis", vmin=v_min, vmax=v_max)
    axes[1].set_title("adjusted")

    axes[0].axis("off")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ================================================================
# Display a chosen FOV at a time relative to the start
# ================================================================

import datetime
import re

# INPUT image adjustment parameters

fov_to_display = 1
# v_min_perc = [0]
# v_max_perc = [100]
# v_min = np.percentile(img, v_min_perc[i])
# v_max = np.percentile(img, v_max_perc[i])

v_min = 0
v_max = 50

time_from_start_min = 480 # minutes from the first frame in the FOV

save = True
save_output_dir = output_path




timestamp_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}-\d{6})")
timestamp_format = "%Y-%m-%d_%H-%M-%S-%f"

def _get_timestamp(filename: str) -> datetime.datetime:
    match = timestamp_pattern.search(Path(filename).name)
    if match is None:
        raise ValueError(f"No timestamp found in {filename}")
    return datetime.datetime.strptime(match.group(1), timestamp_format)

def _sort_by_time(file_list):
    return sorted(file_list, key=_get_timestamp)

def _select_frame_at_relative_time(file_list, time_from_start_min: float):
    ordered_files = _sort_by_time(file_list)
    timestamps = [_get_timestamp(f) for f in ordered_files]
    start_time = timestamps[0]
    target_time = start_time + datetime.timedelta(minutes=float(time_from_start_min))
    deltas = [abs((ts - target_time).total_seconds()) for ts in timestamps]
    frame_idx = int(np.argmin(deltas))
    actual_relative_min = (timestamps[frame_idx] - start_time).total_seconds() / 60.0
    time_error_min = deltas[frame_idx] / 60.0
    return ordered_files[frame_idx], timestamps[frame_idx], actual_relative_min, time_error_min

print(f"Plotting FOV no {fov_to_display} at {time_from_start_min:.2f} min from start")

# Display cropped images with raw and adjusted contrast per LED

for i, led in enumerate(sel_led):
    if fov_to_display not in filenames_by_fov_by_led[led]:
        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].set_axis_off()
        axes[1].set_axis_off()
        fig.suptitle(led)
        axes[0].set_title("FOV not found")
        plt.tight_layout()
        plt.show()
        continue

    first_file, actual_time, actual_relative_min, time_error_min = _select_frame_at_relative_time(
        filenames_by_fov_by_led[led][fov_to_display],
        time_from_start_min,
    )
    img = io.imread(first_file)[y_min:y_max, x_min:x_max]

    fig, axes = plt.subplots(1, 2, figsize=(4, 5))
    fig.suptitle(
        f"{led} | target={time_from_start_min:.2f} min | actual={actual_relative_min:.2f} min \n frame={actual_time.strftime('%Y-%m-%d %H:%M:%S')} | error={time_error_min:.2f} min"
    )
    axes[0].imshow(img, cmap="viridis")
    axes[0].set_title("raw")

    axes[1].imshow(img, cmap="viridis", vmin=v_min, vmax=v_max)
    axes[1].set_title("adjusted")

    axes[0].axis("off")
    axes[1].axis("off")

    if save:
        save_output_dir.mkdir(parents=True, exist_ok=True)
        save_name = f"{experiment_name}___{led}___FOV{fov_to_display}___t{time_from_start_min:.2f}min_adjusted.png".replace(" ", "_")
        save_fig, save_ax = plt.subplots(figsize=(6, 6))
        save_ax.imshow(img, cmap="viridis", vmin=v_min, vmax=v_max)
        save_ax.axis("off")
        save_ax.set_title(
            f"{led} | target={time_from_start_min:.2f} min | actual={actual_relative_min:.2f} min | error={time_error_min:.2f} min"
        )
        save_fig.tight_layout()
        save_fig.savefig(save_output_dir / save_name, dpi=600, bbox_inches="tight", pad_inches=0.02)
        plt.close(save_fig)

    plt.tight_layout()
    plt.show()